### Configuração de Ambiente e Segurança (Setup)

> Este notebook é responsável pela inicialização segura do ambiente de desenvolvimento. O objetivo principal é mapear as credenciais de infraestrutura sem expor dados sensíveis no código-fonte, aderindo às melhores práticas de segurança da informação.

**Decisões Arquiteturais:**
1. **Gestão de Segredos:** A biblioteca `python-dotenv` executa uma busca dinâmica na árvore de diretórios para localizar e carregar o arquivo `.env` de forma resiliente, independentemente do nível da pasta em que o notebook seja executado.
2. **Compatibilidade Serverless:** A injeção global de credenciais no cluster através do comando `spark.conf.set()` foi **intencionalmente omitida**. Esta decisão evita falhas de execução (`[JVM_ATTRIBUTE_NOT_SUPPORTED]`), garantindo total compatibilidade com o protocolo Spark Connect utilizado pela arquitetura Databricks Serverless. As credenciais mapeadas aqui são repassadas sob demanda (via parâmetros `storage_options`) nas camadas de Ingestão e Análise.

In [0]:
import os
from dotenv import load_dotenv

print("⚙️ Inicializando o módulo de configuração de ambiente...")

# 1. BUSCA DINÂMICA PELO ARQUIVO .ENV
# O sistema testa múltiplos níveis de diretório para garantir resiliência
caminho_env = None
for tentativa in [".env", "../.env", "../../.env"]:
    if os.path.exists(tentativa):
        caminho_env = tentativa
        break

if caminho_env:
    load_dotenv(dotenv_path=caminho_env)
    print(f"✅ Arquivo .env localizado e carregado com sucesso pelo sistema a partir de: '{caminho_env}'")
else:
    raise FileNotFoundError("⚠️ ERRO CRÍTICO: O sistema não localizou o arquivo .env de credenciais.")

# 2. MAPEAMENTO DAS VARIÁVEIS DE INFRAESTRUTURA
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = "internshipdatalake"

# Validação básica de integridade das chaves
if not all([client_id, tenant_id, client_secret]):
    print("⚠️ Aviso: O sistema detectou que algumas credenciais do Azure Data Lake estão ausentes no arquivo .env.")
else:
    print("🔒 Credenciais do Azure Data Lake (ADLS) validadas e mapeadas na memória do cluster com segurança.")

# FIM DO NOTEBOOK DE SETUP.